# PhoPunct Inference — Nạp checkpoint và thử suy luận trên câu tùy chỉnh

Notebook này nạp checkpoint đã train của **PhoPunct (model #17: PhoBERT-large + BiLSTM + CRF)**
và chạy suy luận (thêm dấu câu) trên các câu văn bản tiếng Việt **chưa có dấu câu**.

**Yêu cầu đầu vào (theo đúng ràng buộc đã xác nhận trong project):**
- Câu phải là tiếng Việt **có dấu (accented)**, không phải romanized/không dấu.
- Các từ trong câu nên đã **tách từ giống lúc train** (mỗi từ/token cách nhau bởi khoảng trắng,
  từ ghép nhiều âm tiết nối bằng `_` nếu dữ liệu train dùng cách này — kiểm tra lại 1 dòng trong
  `data/punctuation/News/train.txt` nếu không chắc từ có bị nối `_` hay không).
- Model được train với `max_seq_length=256` (subword) — câu quá dài sẽ bị cắt bớt, notebook sẽ
  cảnh báo nếu xảy ra.

**Trước khi chạy:** sửa các biến trong ô "CONFIG" bên dưới cho khớp với máy Fa.

## 1. Cấu hình đường dẫn (sys.path để import PhoBertLstmCrf/punc_dataset_word)

In [1]:
import sys
import os

# --------------------------------------------------------------------- #
# CONFIG - SỬA CÁC DÒNG DƯỚI ĐÂY CHO KHỚP VỚI MÁY FA
# --------------------------------------------------------------------- #

# Thư mục chứa phobert_lstm_crf.py + punc_dataset_word.py (script train gốc)
SCRIPT_DIR = r"D:\COLING2027\2026_08_09\phopunct"

# Checkpoint muốn nạp - đổi giữa News/Novels tùy ý
CHECKPOINT_PATH = r"D:\COLING2027\2026_08_09\phopunct\outputs_from_gpu\phopunct_novels\best_checkpoint.pt"
# CHECKPOINT_PATH = r"D:\COLING2027\2026_08_09\phopunct\outputs_from_gpu\phopunct_novels\best_checkpoint.pt"

# Backbone HuggingFace dùng lúc train (PhoPunct = phobert-large)
MODEL_NAME = "vinai/phobert-large"

# "cuda" nếu máy có GPU + đã cài torch bản CUDA, "cpu" nếu chạy CPU (chậm hơn nhưng vẫn chạy được)
DEVICE = "cpu"

MAX_SEQ_LENGTH = 256
LSTM_HIDDEN_SIZE = 128   # phải khớp với lúc train (mặc định trong phobert_lstm_crf.py)

# --------------------------------------------------------------------- #

sys.path.insert(0, SCRIPT_DIR)
assert os.path.isfile(CHECKPOINT_PATH), f"Không tìm thấy checkpoint: {CHECKPOINT_PATH}"
print("OK - sys.path và checkpoint đã sẵn sàng.")


OK - sys.path và checkpoint đã sẵn sàng.


## 2. Nạp tokenizer + model từ checkpoint

In [2]:
import torch

from phobert_lstm_crf import PhoBertLstmCrf
from punc_dataset_word import LABELS, ID2LABEL, LABEL2ID
from transformers import AutoTokenizer

device = torch.device(DEVICE)

print("Đang nạp tokenizer (lần đầu có thể mất thời gian tải về nếu chưa có cache)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print("Đang khởi tạo model + nạp checkpoint...")
model = PhoBertLstmCrf(MODEL_NAME, num_labels=len(LABELS), lstm_hidden_size=LSTM_HIDDEN_SIZE).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Nạp xong. Checkpoint từ epoch {ckpt.get('epoch')}, best_f1 lúc lưu = {ckpt.get('best_f1'):.4f}")


c:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang nạp tokenizer (lần đầu có thể mất thời gian tải về nếu chưa có cache)...


08/10/2026 14:10:36 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/vinai/phobert-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
08/10/2026 14:10:36 - WARNING - huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
08/10/2026 14:10:36 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-large/70e2cfcd3cce29c970aee4954ea34a32bb30afdc/config.json "HTTP/1.1 200 OK"
08/10/2026 14:10:36 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/vinai/phobert-large/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
08/10/2026 14:10:36 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/vinai/phobert-large/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
08/10/2026 14:10:36 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-large/tree/main/additional

Đang khởi tạo model + nạp checkpoint...


08/10/2026 14:10:38 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/vinai/phobert-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
08/10/2026 14:10:38 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-large/70e2cfcd3cce29c970aee4954ea34a32bb30afdc/config.json "HTTP/1.1 200 OK"
08/10/2026 14:10:39 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/vinai/phobert-large/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
08/10/2026 14:10:39 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-large "HTTP/1.1 200 OK"
08/10/2026 14:10:40 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-large/commits/main "HTTP/1.1 200 OK"
08/10/2026 14:10:40 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-large/discussions?p=0 "HTTP/1.1 200 OK"
08/10/2026 14:10:41 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/

Nạp xong. Checkpoint từ epoch 6, best_f1 lúc lưu = 0.4897


## 3. Hàm suy luận (punctuate)

Nhận câu chưa có dấu câu (các từ cách nhau bởi khoảng trắng), trả về câu đã được chèn dấu câu.
Có tùy chọn viết hoa chữ cái đầu câu / sau dấu kết câu (`.`/`?`/`!`) cho tự nhiên hơn.

In [3]:
PUNCT_MAP = {
    "O": "",
    "PERIOD": ".",
    "COMMA": ",",
    "COLON": ":",
    "QMARK": "?",
    "EXCLAM": "!",
    "SEMICOLON": ";",
}
SENTENCE_END_LABELS = {"PERIOD", "QMARK", "EXCLAM"}


def _encode_words_for_inference(words, tokenizer, max_seq_length):
    """Giống hệt logic gather word-level trong punc_dataset_word.convert_examples_to_features,
    nhưng không cần label (dùng cho inference)."""
    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    budget = max_seq_length - 2
    subword_ids, word_starts, kept_words = [], [], []
    truncated = False

    for w in words:
        piece_ids = tokenizer.encode(w, add_special_tokens=False)
        if not piece_ids:
            continue
        if len(subword_ids) + len(piece_ids) > budget:
            truncated = True
            break
        subword_ids.extend(piece_ids)
        word_starts.extend([1] + [0] * (len(piece_ids) - 1))
        kept_words.append(w)

    input_ids = [bos_id] + subword_ids + [eos_id]
    word_starts_full = [0] + word_starts + [0]
    attention_mask = [1] * len(input_ids)

    while len(input_ids) < max_seq_length:
        input_ids.append(pad_id)
        attention_mask.append(0)
        word_starts_full.append(0)

    word_mask = [1] * len(kept_words)
    while len(word_mask) < max_seq_length:
        word_mask.append(0)

    return input_ids, attention_mask, word_starts_full, word_mask, kept_words, truncated


@torch.no_grad()
def punctuate(text: str, capitalize: bool = True) -> str:
    words = text.strip().split()
    if not words:
        return text

    input_ids, attention_mask, word_starts, word_mask, kept_words, truncated = \
        _encode_words_for_inference(words, tokenizer, MAX_SEQ_LENGTH)

    if truncated:
        print(f"[Cảnh báo] Câu dài hơn max_seq_length={MAX_SEQ_LENGTH} subword, "
              f"đã cắt bớt còn {len(kept_words)}/{len(words)} từ.")

    input_ids_t = torch.tensor([input_ids], dtype=torch.long, device=device)
    attention_mask_t = torch.tensor([attention_mask], dtype=torch.long, device=device)
    word_starts_t = torch.tensor([word_starts], dtype=torch.long, device=device)
    word_mask_t = torch.tensor([word_mask], dtype=torch.long, device=device)

    pred_seqs = model(input_ids_t, attention_mask_t, word_starts_t,
                       label_ids=None, word_mask=word_mask_t)
    pred_labels = [ID2LABEL[i] for i in pred_seqs[0][:len(kept_words)]]

    out_tokens = []
    cap_next = capitalize
    for w, lab in zip(kept_words, pred_labels):
        token = w[0].upper() + w[1:] if (cap_next and w) else w
        cap_next = False
        out_tokens.append(token)
        mark = PUNCT_MAP.get(lab, "")
        if mark:
            out_tokens[-1] = out_tokens[-1] + mark
        if lab in SENTENCE_END_LABELS:
            cap_next = capitalize

    return " ".join(out_tokens)


print("Hàm punctuate() đã sẵn sàng.")


Hàm punctuate() đã sẵn sàng.


## 4. Thử suy luận — sửa danh sách câu bên dưới theo ý Fa

Lưu ý: các câu mẫu bên dưới đã bỏ hết dấu câu, các từ cách nhau bởi khoảng trắng
(chưa nối `_` cho từ ghép — nếu dữ liệu train có nối `_`, Fa nên tự nối lại cho khớp
để có kết quả chính xác nhất).

In [4]:
SENTENCES = [
    "hôm nay trời đẹp quá chúng ta cùng đi chơi nhé",
    "bạn có khỏe không tôi rất nhớ bạn",
    "xin chào tôi là sinh viên năm cuối nghiên cứu về xử lý ngôn ngữ tự nhiên",
    "anh ơi có cần giúp gì không nếu cần cứ gọi tôi bất cứ lúc nào",
]

for s in SENTENCES:
    result = punctuate(s)
    print("Input :", s)
    print("Output:", result)
    print("-" * 80)


Input : hôm nay trời đẹp quá chúng ta cùng đi chơi nhé
Output: Hôm nay trời đẹp quá, chúng ta cùng đi chơi nhé!
--------------------------------------------------------------------------------
Input : bạn có khỏe không tôi rất nhớ bạn
Output: Bạn có khỏe không? Tôi rất nhớ bạn!
--------------------------------------------------------------------------------
Input : xin chào tôi là sinh viên năm cuối nghiên cứu về xử lý ngôn ngữ tự nhiên
Output: Xin chào, tôi là sinh viên năm cuối, nghiên cứu về xử lý ngôn ngữ tự nhiên.
--------------------------------------------------------------------------------
Input : anh ơi có cần giúp gì không nếu cần cứ gọi tôi bất cứ lúc nào
Output: Anh ơi, có cần giúp gì không? Nếu cần cứ gọi tôi, bất cứ lúc nào!
--------------------------------------------------------------------------------


## 5. (Tùy chọn) Thử nhanh 1 câu tự nhập

In [5]:
custom_sentence = "tôi tên là khoa bạn tên là gì nhỉ tôi tên là mai rất vui được quen biết bạn"
print(punctuate(custom_sentence))


Tôi tên là khoa. Bạn tên là gì nhỉ? Tôi tên là mai. Rất vui được quen biết bạn.
